In [1]:
%matplotlib inline

In [2]:
import glob, re
import string
import json
import math

import numpy as np
import pandas as pd

from fuzzywuzzy import fuzz
from typing import Tuple
from pathlib import Path

In [3]:
#!uv pip install python-Levenshtein
#!uv pip install fuzzywuzzy

# Code assigns speaker labels to AA Transcripts

Uses labels from fan-made Uncut Friends Scripts transcripts to time-stamped, non-identified AssemblyAI speech2text transcripts 

In [3]:
"""
Adjust local paths to Uncut Friends Scripts (UFS) and AssemblyAI (AA) transcripts, and output directory here
"""
usf_dir = Path(
    "/home/mstlaure/Documents/Marie/neuromod/friends_annotations/annotation_results/community_based"    
)
aa_dir = Path(
    "/home/mstlaure/Documents/Marie/neuromod/friends_annotations/annotation_results/Speech2Text"    
)

# Change to save output in different directory from 
outdir = aa_dir


In [4]:

def prep_ufs(s_num, e_num):
    """
    Extracts cleaned utterances from 
    Uncut Friends Scripts fan-made transcripts
    """
    ufs_filepath = Path(
        f"{usf_dir}/s{s_num}/friends_s0{s_num}e{e_num}_ufs.txt"
    )
    with open(ufs_filepath, 'r', encoding='utf-8') as f:
        ufs_transcript = f.read()

    ufs_processed = []
    for line in ufs_transcript.splitlines():
        # Remove all text inside parentheses and squared brackets (transcriber notes)
        line = re.sub(r'\s*\(.*?\)\s*', ' ', line)
        line = re.sub(r'\s*\[.*?\]\s*', ' ', line)
        line = line.strip()    
        # Only process lines that start with "Speaker:"
        if ":" in line:
            # Split line between speaker and utterance
            line_segs = line.strip().lower().split(":")
            speaker = line_segs[0]
            speech = line_segs[1]
            # To handle clock times
            if len(line_segs) > 2:    
                for seg in line_segs[2:]:
                    speech += f":{seg}"
                
            sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|\!)\s', speech)
            for s in sentences:
                if s.strip():
                    ufs_processed.append((speaker, s))

    return ufs_processed


def prep_aa(s_num, e_num):
    """
    Load cleaned up AssemblyAI speech2text transcripts
    (theme song removed)
    """
    json_file_path = Path(
        f"{aa_dir}/s{s_num}/friends_s0{s_num}e{e_num}_"
        "model-AA_desc-wUtter_transcript.json"   
    )
    with open(json_file_path, 'r') as file:
        aa_transcript = json.load(file)

    return aa_transcript


def save_aa(s_num, e_num, aa_transcript):
    """
    Load cleaned up AssemblyAI speech2text transcripts
    (theme song removed)
    """
    json_out_path = Path(
        f"{outdir}/s{s_num}/friends_s0{s_num}e{e_num}_"
        "model-AA_desc-wSpeaker_transcript.json"   
    )
    with open(json_out_path, "w") as outfile:
        json.dump(aa_transcript, outfile)    
    

def string_match(
    utter_1: str,
    utter_2: str,
) -> Tuple[int, int]:
    """
    Calculates a resemblance score between two strings.

    Args:
        utter_1: The first string.
        utter_2: The second string.

    Returns:
        A tuple containing:
        - The fuzzywuzzy ratio score (int).
        - The number of characters in the first string (int).
    """
    clean_string = str.maketrans('', '', string.punctuation + string.whitespace)

    clean_u1 = utter_1.lower().translate(clean_string)
    clean_u2 = utter_2.lower().translate(clean_string)

    return fuzz.ratio(clean_u1, clean_u2), len(utter_1)

    
def find_bestmatch(
    utter: str,
    candidates: list[tuple],
    i_0 = None,
    i_n = None,
    verbose = False,
) -> list:
    """
    Compares a string to a list of strings
    Returns closest match index, matching score,
    and first utterance length (to evaluate condifence 
    for filtering)
    """
    top = [-1, -1, None]
    best_match = None
    i_0 = 0 if i_0 is None else i_0
    i_n = len(candidates) if i_n is None else i_n

    for i in range(i_0, i_n):
        utt_i = candidates[i]
        fuzz_r, len_u = string_match(utter, utt_i[1])
        if fuzz_r > top[1]:
            top = [i, fuzz_r, len_u]
            best_match = utt_i[1]
    
    if verbose:
        print(f"{utter} \n {best_match}\n\n")
    
    return top 


def find_anchors(aa_transcript, processed_utterances, verbose=False):
    """
    For each speech2text sentence, find highest match in fan-made transcript.
    Identify high-confidence matches (anchors) to structure the matching process 
    (done in chunks solved recursively)

    Anchors are pairings with > 90% fuzzywuzzy ratio match on 
    utterances long enough to be distinct and unique (i.e., not "Yeah").    
    """
    top_scores = [
        [i] + find_bestmatch(
            x['text'], processed_utterances, verbose=verbose,
        ) for i, x in enumerate(aa_transcript['sentences'])
    ]

    anchors = np.logical_and(
        np.array([x[2] for x in top_scores]) > 93,
        np.array([x[3] for x in top_scores]) > 35,
    )
    # Safety check: disregard anchors where the match idx is lower than the previous anchor's
    anchors = anchors * np.logical_and(
        np.array(
            [True] + [top_scores[i][1] >= top_scores[i-1][1] for i in range(1, len(top_scores))]
        ), 
        np.array(
            [top_scores[i][1] <= top_scores[i+1][1] for i in range(0, len(top_scores)-1)] + [True] 
        ),
    )
    
    return anchors, top_scores


def assign_chunk(
    aa_i0,
    aa_in,
    ufs_i0,
    ufs_in,
    aa_transcript,
    processed_utterances,
    score_assignments,
    verbose=False,
):
    """
    Recursive function finds the closest match 
    for each sentence in a 'chunk' of consecutive 
    sentences within a set range of targets.

    To structure the search (i.e., assign high match first, 
    maintain serial order of assigned targets), the chunk's 
    sentence with the highest matching score is used to 
    split the chunk into two parts. The search is re-launched 
    on both halves, up until chunk size == 1 sentence,
    then the target assignment is saved for that sentence.
    """
    chunk_scores = np.array([
        [aa_i0 + j] + find_bestmatch(
            x['text'],
            processed_utterances,
            ufs_i0,
            ufs_in,
            verbose=verbose,
        ) for j, x in enumerate(aa_transcript['sentences'][aa_i0:aa_in])
    ])

    if chunk_scores.shape[0] == 1:
        score_assignments[aa_i0:aa_in] = chunk_scores
    else:
        # identify next highest match as a new "anchor", exculding the top row (anchor)
        split_idx = int(np.argmax(chunk_scores[1:, 2]) + 1)

        assert chunk_scores[0][0] == aa_i0
        assert aa_i0 + split_idx == chunk_scores[split_idx][0]
        # lauch recursively on top and bottom chunks
        assign_chunk(
            chunk_scores[0][0],              # includes chunk's top sentence
            chunk_scores[split_idx][0],      # excludes chunk's split sentence
            chunk_scores[0][1],              # includes chunk's top sentence's match
            chunk_scores[split_idx][1] + 1,  # include chunk's split sentence match, in case precedent also matches that utterance
            aa_transcript,
            processed_utterances,
            score_assignments,
        )
        assign_chunk(
            chunk_scores[split_idx][0],      # includes split sentence
            chunk_scores[-1][0] + 1,         # includes chunk's last sentence
            chunk_scores[split_idx][1],      # includes split sentence's match
            ufs_in,                          # includes lower edge of match search space
            aa_transcript,
            processed_utterances,
            score_assignments,
        )
            

In [5]:

def process_chunks(
    ancs,
    tops,
    aa_transcript,
    processed_utterances,
    verbose=False,
):
    """
    Assign a match to each sentence in the transcript. 

    Breaks down sentences into chunks bound by "anchors", 
    which are very high confidence matches. Solves each
    chunk recursively by calling assign_chunk. 

    Search for (upper/lower) boundary in target space 
    for top/bottom sentences unbound by "anchors".
    """
    score_assignments = np.full(
        (len(tops), 4), -1, dtype=int,
    )
    anchor_idx = np.array(tops)[ancs]    
    
    for i in range(anchor_idx.shape[0]): 
        
        if i == anchor_idx.shape[0] - 1:
            """
            Assign sentences from the last anchor
            to the last sentence
            """
            j_n = forward2btm(
                anchor_idx[-1], 
                aa_transcript, 
                processed_utterances,
            )
            assign_chunk(
                anchor_idx[-1][0],                      # include last anchor speech-to-text in search
                len(aa_transcript['sentences']),        # process all remaining sentences
                anchor_idx[-1][1],                      # include last anchor's ccf match in search 
                j_n + 1,                                # include last sentence's ccf match in search
                aa_transcript,
                processed_utterances, 
                score_assignments,
                verbose=verbose,
            )
            s_n, l_n = string_match(
                aa_transcript['sentences'][-1]['text'],
                processed_utterances[j_n][1],
            )
            score_assignments[-1] = np.array([len(tops)-1, j_n, s_n, l_n])
        
        else:
            assign_chunk(
                anchor_idx[i][0],          # include current anchor speech-to-text in search
                anchor_idx[i+1][0],        # exclude next anchor speech-to-text from search
                anchor_idx[i][1],          # include current anchor's ccf match in search 
                anchor_idx[i+1][1] + 1,    # include next anchor's ccf match in search, in case precedent text also matches that utterance  
                aa_transcript,
                processed_utterances, 
                score_assignments,
                verbose=verbose,
            )
            
    """
    Assign sentences before the first anchor, if any
    """
    if anchor_idx[0][0] > 0:
        j_0 = backtrack2top(
            anchor_idx[0],
            aa_transcript,
            processed_utterances, 
        )
        assign_chunk(
            0,                        # include first speech-to-text sentence in search
            anchor_idx[0][0],         # exclude first anchor speech-to-text from search
            j_0,                      # include first sentence's ccf match in search 
            anchor_idx[0][1] + 1,     # include first anchor's ccf match in search, in case precedent text also matches that utterance  
            aa_transcript,
            processed_utterances, 
            score_assignments,
            verbose=verbose,
        )
        s_0, l_0 = string_match(
            aa_transcript['sentences'][0]['text'],
            processed_utterances[j_0][1],
        )
        score_assignments[0] = np.array([0, j_0, s_0, l_0])
    
    return anchor_idx, score_assignments


def backtrack2top(
    start_vals,   # anchor_idx[0]
    aa_transcript,
    processed_utterances, 
):
    """
    Starting from the first anchor, work backward to find the 
    upper boundary (first sentence's match) for the AA transcript 
    (partial episode) in the community transcript (full episode).
    """
    i, j, _, _ = start_vals
    if i == 0:
        """
        Edge case for which the first anchor is also the first speech2text sentence
        No unassigned sentences remain at the top, nothing to do.
        """
        return j
        
    else:
        # Concatenate all text from top sentences (including first anchor)
        aa_toptext = " ".join([s['text'] for s in aa_transcript['sentences'][:i+1]])
        # concat community transcript text to find optimal match 
        top_score = (-1, -1)
        target_text = processed_utterances[j][1]
        
        for k in range(j-1, -1, -1): 
            target_text = " ".join([processed_utterances[k][1], target_text])
            score = (k, string_match(aa_toptext, target_text)[0])            
            if score[1] > top_score[1]:
                top_score = score

        """
        Tweak first target (upper boundary) to improve fit 
        Can prevent crashing when episode starts mid-sentence
        """
        target_text = " ".join([u[1] for u in processed_utterances[top_score[0]:j+1]])
        if top_score[0] > 0:
            buffer_score = (0, top_score[1])
            buffer_words = processed_utterances[top_score[0] - 1][1].strip().split(" ")
            buffer_speaker = processed_utterances[top_score[0] - 1][0]
            for m in range(-1, -len(buffer_words) - 1, -1): 
                target_text = " ".join([buffer_words[m], target_text])
                score = (m, string_match(aa_toptext, target_text)[0])            
                if score[1] > buffer_score[1]:
                    buffer_score = score

            if buffer_score[0] < 0:
                if processed_utterances[top_score[0]][0] != buffer_speaker:
                    processed_utterances[top_score[0]-1] = (
                        buffer_speaker, 
                        " ".join(buffer_words[buffer_score[0]:]),
                    )
                    return top_score[0]-1
                else:
                    processed_utterances[top_score[0]] = (
                        processed_utterances[top_score[0]][0], 
                        " ".join([" ".join(buffer_words[buffer_score[0]:]), processed_utterances[top_score[0]][1]])
                    )

        return top_score[0]


def forward2btm(
    start_vals,   # anchor_idx[-1]
    aa_transcript,
    processed_utterances, 
):
    """
    Starting from the last anchor, work backward to find the 
    lower boundary (last sentence match) of the AA transcript 
    (partial episode) in the community transcript (full episode).
    """
    i, j, _, _ = start_vals
    if i == len(aa_transcript['sentences']) - 1:
        """
        Edge case for which the last anchor is also the last speech2text sentence.
        No unassigned sentences remain below (just assign last anchor to score array).
        """
        return j
        
    else:
        # Concatenate all text from last anchor to last sentence (both included)
        aa_toptext = " ".join([s['text'] for s in aa_transcript['sentences'][i:]])

        # concat community transcript text to find optimal match 
        top_score = (-1, -1)
        target_text = processed_utterances[j][1]
        
        for k in range(j+1, len(processed_utterances), 1): 
            target_text = " ".join([target_text, processed_utterances[k][1]])
            score = (k, string_match(aa_toptext, target_text)[0])            
            if score[1] > top_score[1]:
                top_score = score

        """
        Tweak last target (lower boundary) to improve fit 
        Can prevent mis-labelling when episode ends mid-sentence
        """
        target_text = " ".join([u[1] for u in processed_utterances[j:top_score[0]+1]])
        if top_score[0] + 1 < len(processed_utterances):
            buffer_score = (-1, top_score[1])
            buffer_words = processed_utterances[top_score[0] + 1][1].strip().split(" ")
            buffer_speaker = processed_utterances[top_score[0] + 1][0]
            for m in range(len(buffer_words)): 
                target_text = " ".join([target_text, buffer_words[m]])
                score = (m, string_match(aa_toptext, target_text)[0])            
                if score[1] > buffer_score[1]:
                    buffer_score = score

            if buffer_score[0] > -1:
                if processed_utterances[top_score[0]][0] != buffer_speaker:
                    processed_utterances[top_score[0]+1] = (
                        buffer_speaker, 
                        " ".join(buffer_words[:buffer_score[0]+1]),
                    )
                    return top_score[0]+1
                else:
                    processed_utterances[top_score[0]] = (
                        processed_utterances[top_score[0]][0], 
                        " ".join([processed_utterances[top_score[0]][1], " ".join(buffer_words[:buffer_score[0]+1])])
                    )

        return top_score[0]



In [6]:

def merge_segs(aa_t, i):
    """
    Merge two adjacent sentences into a single one
    """
    if i + 1 < len(aa_t['sentences']) and i > -1:
        s1 = aa_t['sentences'][i]
        s2 = aa_t['sentences'][i+1]
        new_sentence = {
            'text': s1['text'] + " " + s2['text'],
            'start': s1['start'],
            'end': s2['end'],
            'speaker': s1['speaker'],
            'confidence': None,
            'words': s1['words'] + s2['words'],
        }

        last_seg = [] if i + 2 == len(aa_t['sentences']) else aa_t['sentences'][i+2:]
        aa_t['sentences'] = aa_t['sentences'][:i] + [new_sentence] + last_seg
                
    return aa_t


def jitter_segs(aa_t, i, split_idx):
    """
    Re-distributes words in-place between adjacent sentences
    """
    temp_wlist = aa_t['sentences'][i]['words'] + aa_t['sentences'][i+1]['words']
    
    aa_t['sentences'][i]['words'] = temp_wlist[:split_idx]
    aa_t['sentences'][i]['text'] = " ".join([w['word'] for w in temp_wlist[:split_idx]])
    aa_t['sentences'][i]['end'] = aa_t['sentences'][i]['words'][-1]['end']

    aa_t['sentences'][i+1]['words'] = temp_wlist[split_idx:]
    aa_t['sentences'][i+1]['text'] = " ".join([w['word'] for w in temp_wlist[split_idx:]])
    aa_t['sentences'][i+1]['start'] = aa_t['sentences'][i+1]['words'][0]['start']

    return aa_t
    

def split_seg(aa_t, i, w2drop):
    """
    Split a sentence into two sentences.
    """
    wlist = aa_t['sentences'][i]["words"]
    new_sentence = {
        'text': " ".join([w['word'] for w in wlist[:w2drop]]),
        'start': wlist[0]['start'],
        'end': wlist[w2drop-1]['end'],
        'speaker': None,
        'confidence': None,
        'words': wlist[:w2drop],
    }
    
    aa_t['sentences'][i]['text'] = " ".join([w['word'] for w in wlist[w2drop:]])    
    aa_t['sentences'][i]['start'] = wlist[w2drop]['start']    
    aa_t['sentences'][i]['words'] = wlist[w2drop:]

    aa_t['sentences'] = aa_t['sentences'][:i] + [new_sentence] + aa_t['sentences'][i:]
        
    return aa_t
    

def find_best_split(wlist, target1, target2, i_start=None, i_stop=None):
    """
    Split contatenates list of words and test the cummulative matching score
    with two adjacent target utterances.

    Return the split index with the highest matching score and the score itself
    """
    i_start = 0 if i_start is None else i_start
    i_stop = len(wlist)+1 if i_stop is None else i_stop

    scores = []
    for i in range(i_start, i_stop):
        if i == 0:
            scores.append((i, string_match(
                " ".join([w['word'] for w in wlist]), target2,
            )[0]))        
        elif i == len(wlist):
            scores.append((i, string_match(
                " ".join([w['word'] for w in wlist]), target1,
            )[0]))
        else:
            scores.append((i, string_match(
                " ".join([w['word'] for w in wlist[:i]]),
                target1,
            )[0] + string_match(
                " ".join([w['word'] for w in wlist[i:]]),
                target2,
            )[0]))

    split_idx, best_score = scores[np.argmax([x[1] for x in scores])]
    
    return split_idx, best_score    


def find_words2drop(wlist, target, fromleft=True):
    """
    Determine the number of words to drop from sentence edge 
    to maximize target match
    """
    scores = []
    if fromleft:
        for i in range(len(wlist)):
            scores.append((i, string_match(
                " ".join([w['word'] for w in wlist[i:]]),
                target,
            )[0]))
    else:
        for i in range(len(wlist)):
            scores.append((i, string_match(
                " ".join([w['word'] for w in wlist[:len(wlist)-i]]),
                target,
            )[0]))
            
    words2drop, best_score = scores[np.argmax([x[1] for x in scores])]
    
    return words2drop
    
    
def split_first_seg(aa_t, p_utter, i, j1, j2):
    """
    Optimize the first two sentences' fit by adjusting how they are 
    split and attributed to their targets
    """
    split = find_words2drop(
        aa_t['sentences'][i]['words'] + aa_t['sentences'][i+1]['words'], 
        p_utter[j1][1],
        fromleft=False,
    )
    
    if split == 0:
        aa_t = merge_segs(aa_t, i)
        return aa_t, (j1,)
    else:
        # Optimize split between segs for first sentence to match j1
        flip_split = len(
            aa_t['sentences'][i]['words'] + aa_t['sentences'][i+1]['words']
        ) - split
        aa_t = jitter_segs(aa_t, i, flip_split)
        
        if j2 - j1 < 2:
            return aa_t, (j1, j1+1)
        else:
            l_split = find_words2drop(
                aa_t['sentences'][i+1]['words'], 
                p_utter[j2][1],
            )
            if l_split == 0:
                return aa_t, (j1, j2)
            else:
                aa_t = split_seg(aa_t, i+1, l_split)
                return aa_t, (j1, j1+1, j2)
                

def split_last_seg(aa_t, p_utter, i, j1, j2):
    """
    Optimize the last two sentences' fit by adjusting how they are 
    split and attributed to their targets
    """
    split = find_words2drop(
        aa_t['sentences'][i]['words'] + aa_t['sentences'][i+1]['words'], 
        p_utter[j2][1],
    )
    
    if split == 0:
        aa_t = merge_segs(aa_t, i)
        return aa_t, (j2,)
    else:
        # Optimize split between segs for last sentence to match j2
        aa_t = jitter_segs(aa_t, i, split)
        if j2 - j1 < 2:
            return aa_t, (j2-1, j2)
        else:
            r_split = find_words2drop(
                aa_t['sentences'][i]['words'], 
                p_utter[j1][1],
                fromleft=False,
            )
            if r_split == 0:
                return aa_t, (j1, j2)
            else:
                aa_t = split_seg(
                    aa_t, i,
                    len(aa_t['sentences'][i]['words']) - r_split,
                )
                return aa_t, (j1, j2-1, j2)
                            
    
def adjacent_segs(aa_t, p_utter, i, j):
    """
    Adjust split between adjacent sentences for optimal match
    Note: obsolete, use gap_segs instead, works better w larger context window
    """
    if i + 1 < len(aa_t['sentences']):
        
        # find optimal split between two consecutive sentences
        temp_wlist = aa_t['sentences'][i]['words'] + aa_t['sentences'][i+1]['words']
        split_idx, best_score = find_best_split(
            temp_wlist, 
            p_utter[j][1], 
            p_utter[j+1][1],
        )

        if split_idx == 0: 
            # Assign all combined words to j+1 match
            aa_t = merge_segs(aa_t, i)
            return aa_t, (j+1,)
            
        elif split_idx == len(temp_wlist):
            # Assign all combined words to j match
            aa_t = merge_segs(aa_t, i)
            return aa_t, (j,)
            
        elif split_idx != len(aa_t['sentences'][i]['words']):
            # re-distribute words between adjacent sentences
            aa_t = jitter_segs(aa_t, i, split_idx)
            
    return aa_t, (j, j+1)


def gap_segs(aa_t, p_utter, i, j1, j2):
    """
    Re-splits consecutive sentences to match the gap size 
    between their respective targets to improve alignment
    """
    # concat four consecutive sentences for more context
    i_0 = max(0, i - 1)
    i_n = min(i+3, len(aa_t['sentences']))
    temp_wlist = [w for s in aa_t['sentences'][i_0:i_n] for w in s["words"]]

    # Set search space boundaries: limit split placements to two middle sentences i and i+1
    i_start = 0 if i == 0 else len(aa_t['sentences'][i_0]['words'])
    i_stop = len([w for s in aa_t['sentences'][i_0:i+2] for w in s["words"]])
    
    """
    Find best split for each target between j1 and j2
    """
    j_0 = max(0, j1-1)
    j_n = min(j2+2, len(p_utter))
    split_indices = []
    for k in range(1, j2-j1+1):
        targ1 = " ".join([u[1] for u in p_utter[j_0:j1+k]])
        targ2 = " ".join([u[1] for u in p_utter[j1+k:j_n]])

        split_idx, best_score = find_best_split(
            temp_wlist, 
            targ1, 
            targ2,
            i_start=i_start, 
            i_stop=i_stop + 1,
        )    
        split_indices.append((k, split_idx, best_score))

    # merge two middle sentences
    aa_t = merge_segs(aa_t, i)

    
    # Perform splits and save their target (j_vals)
    prev_cut = i_start
    j_vals = []
    n_cuts = 0
    for k, s, b in split_indices:
        if s > prev_cut and s != i_stop:
            aa_t = split_seg(aa_t, i+n_cuts, s-prev_cut)
            j_vals.append(j1+k-1)
            prev_cut = s
            n_cuts += 1
    j_vals.append(j2)    
     
    return aa_t, j_vals
    

In [7]:

def print_gapinfo(
    i1, 
    i2, 
    j1, 
    j2, 
    aa_t, 
    p_utter,    
):
    """
    Helps debugging: prints info about sentences 
    and their non-consecutive targets
    """
    print(aa_t['sentences'][i1]['text'])
    print(aa_t['sentences'][i2]['text'])
    print()
    for m in range (j1, j2+1):
        print(p_utter[m][1])
    print()
    print()
        

def finetune_segments(
    aa_transcript,
    processed_utterances,
    score_assignments,
    verbose=False,
):
    """
    Adjust sentence boundaries to increase alignment with target utterances

    Changes are made in-place in the transcript dictionary (sentences and 
    their word lists are split and merged to optimize the fit)
    """
    i = 0
    #i = anchor_idx[0][0]
    while i + 1 < len(aa_transcript['sentences']):
        #print(i)
        if i == 0:
            # Handles first two segments
            j1 = score_assignments[i][1]
            j2 = score_assignments[i+1][1]

            aa_transcript, j_vals = split_first_seg(
                aa_transcript, processed_utterances, i, j1, j2,
            )

            temp_scores = []
            for k in range(len(j_vals)):
                s, l = string_match(
                    aa_transcript['sentences'][i+k]['text'],
                    processed_utterances[j_vals[k]][1],
                )
                temp_scores.append([i+k, j_vals[k], s, l])

            score_assignments[i+2:, 0] = score_assignments[i+2:, 0] + len(j_vals) - 2
            score_assignments = np.concatenate(
                (np.array(temp_scores), score_assignments[i+2:])
            )

            i += len(j_vals) - 1
            #i += 1
        
        elif i + 1 == len(aa_transcript['sentences']) - 1:
            # Handles last two segments
            j1 = score_assignments[i][1]
            j2 = score_assignments[i+1][1]
            
            aa_transcript, j_vals = split_last_seg(
                aa_transcript, processed_utterances, i, j1, j2)
                
            temp_scores = []
            for k in range(len(j_vals)):
                s, l = string_match(
                    aa_transcript['sentences'][i+k]['text'],
                    processed_utterances[j_vals[k]][1],
                )
                temp_scores.append([i+k, j_vals[k], s, l])
            
            score_assignments = np.concatenate(
                (score_assignments[:i], np.array(temp_scores))
            )

            i += len(j_vals)
            
        else:
            j1 = score_assignments[i][1]
            j2 = score_assignments[i+1][1]
            
            # two subsequent sentences w same match: merge sentences, or one with bad match
            if j1 >= j2:
                if j1 > j2 and verbose:
                    print(f"Warning! Misordered labels: sentence {i} is assigned to {j1}, Sentence {i+1} is assigned to {j2}")

                aa_transcript = merge_segs(
                    aa_transcript, i,
                )
                s, l = string_match(
                    aa_transcript['sentences'][i]['text'],
                    processed_utterances[j1][1],
                )
                score_assignments[i] = np.array([i, j1, s, l])
                if i + 2 == len(aa_transcript['sentences']):
                    score_assignments = score_assignments[:-1]
                else:
                    score_assignments[i+2:, 0] = score_assignments[i+2:, 0] - 1
                    score_assignments = np.concatenate(
                        (score_assignments[:i+1], score_assignments[i+2:])
                    )
                # i+= 0  # Do NOT update i, sentence list shortened by one 
    
            # Normally covers all other scenarios
            elif j2 > j1:
                
                if j2 - j1 > 2:
                    if verbose:
                        # Handle large gaps... merge two next consecutive segments, assign second seg's match to scores, then process
                        print(f"Warning! Large jump of {j2 - j1} in matching indices between lines {i} and {i+1}")
                        
                        print_gapinfo(
                            i, i+1, j1, j2, 
                            aa_transcript, 
                            processed_utterances,
                        )
                    # Merge i+1 with next sentence to broaden context
                    if i + 3 < len(aa_transcript['sentences']):
                        j2 = score_assignments[i+2][1]
                        aa_transcript = merge_segs(
                            aa_transcript, i+1,    # merge i+1 et i+2
                        )
                        s, l = string_match(
                            aa_transcript['sentences'][i+1]['text'],
                            processed_utterances[j2][1],
                        )
                        score_assignments[i+1] = np.array([i+1, j2, s, l])
                        score_assignments[i+3:, 0] = score_assignments[i+3:, 0] - 1
                        score_assignments = np.concatenate(
                            (score_assignments[:i+2], score_assignments[i+3:])
                        )
                            
                    # Merge i with previous sentence to broaden context
                    if i > 1:
                        i -= 1
                        j1 = score_assignments[i][1]
                        aa_transcript = merge_segs(
                            aa_transcript, i,   # merge i-1 et i
                        )
                        s, l = string_match(
                            aa_transcript['sentences'][i]['text'],
                            processed_utterances[j1][1],
                        )
                        score_assignments[i] = np.array([i, j1, s, l])

                        score_assignments[i+2:, 0] = score_assignments[i+2:, 0] - 1
                        score_assignments = np.concatenate(
                            (score_assignments[:i+1], score_assignments[i+2:])
                        )
                    
                aa_transcript, j_vals = gap_segs(
                    aa_transcript, processed_utterances, i, j1, j2,
                )
                temp_scores = []
                for k in range(len(j_vals)):
                    s, l = string_match(
                        aa_transcript['sentences'][i+k]['text'],
                        processed_utterances[j_vals[k]][1],
                    )
                    temp_scores.append([i+k, j_vals[k], s, l])
                if i + 2 == len(aa_transcript['sentences']):
                    score_assignments = np.concatenate(
                        (score_assignments[:i], np.array(temp_scores))
                    )
                else:
                    score_assignments[i+2:, 0] = score_assignments[i+2:, 0] + len(j_vals) - 2
                    score_assignments = np.concatenate(
                        (score_assignments[:i], np.array(temp_scores), score_assignments[i+2:])
                    )
                i += len(j_vals) - 1    
                
            else:
                i + 1
                
    return aa_transcript, score_assignments



In [8]:

def give_speaker_labels(
    aa_t,
    p_utter,
    score_assignments,    
):
    """
    Assigns speaker labels from UFS transcripts to 
    AA transcripts (global word list, sentences, and sentence word lists) 
    after the alignment is complete.
    """
    word_list = aa_t['words']

    # Perform sanity checks
    swords_list = [w for s in aa_t['sentences'] for w in s["words"]]
    assert word_list == swords_list
    assert aa_t['transcript'].split(" ") == [w['word'] for w in word_list]
    assert len(word_list) == len(swords_list)
    assert len(aa_t['sentences']) == score_assignments.shape[0]
    
    w_idx = 0    
    for i in range(len(aa_t['sentences'])):

        i_sa, j_sa, _, _ = score_assignments[i]
        assert i_sa == i

        speaker_name = p_utter[j_sa][0]
        
        sen = aa_t['sentences'][i]
        assert sen['text'].split(" ") == [w['word']for w in sen['words']]
        assert sen['text'] == " ".join([w['word']for w in sen['words']])
        num_words = len(sen['words'])
        assert sen['words'] == word_list[w_idx:w_idx+num_words]

        aa_t['sentences'][i]['speaker'] = speaker_name
        for w in sen['words']:
            w['speaker'] = speaker_name
        for w in word_list[w_idx:w_idx+num_words]:
            w['speaker'] = speaker_name
        
        w_idx += num_words

    return aa_t


def build_final_sentences(
    aa_t,
):
    """
    Merge consecutive sentence bits from same speaker if spoken within short time frame
    (as same utterance)
    """
    
    i = 0
    while i < len(aa_t['sentences']) - 1:
        speak1 = aa_t['sentences'][i]['speaker']
        speak2 = aa_t['sentences'][i+1]['speaker']

        off_s1 = aa_t['sentences'][i]['end']
        on_s2 = aa_t['sentences'][i+1]['start']

        if speak1 == speak2 and on_s2 - off_s1 < 1.5:
            aa_t = merge_segs(aa_t, i)
        else: 
            i += 1
        
    return aa_t


In [9]:
"""
Main function to be called in a loop on each transcript
"""

def assign_speakers(s, e):
    """
    s (str) season, e.g., "1", "3"
    e (str) episode, e.g., "02a", "23c"
    """
    processed_ufs = prep_ufs(s, e[:2])
    transcript_aa = prep_aa(s, e)

    # Identify long segments w high confidence matches
    ancs, tops = find_anchors(
        transcript_aa,
        processed_ufs,
        verbose=True,
    )

    # Assign transcribed segments to utterances
    anchor_idx, score_assignments = process_chunks(
        ancs,
        tops,
        transcript_aa,
        processed_ufs,   
        verbose=True,
    )

    # Iterates until no more jumps
    for c in range(6):
        transcript_aa, score_assignments = finetune_segments(
            transcript_aa,
            processed_ufs,   
            score_assignments,
            verbose=True,
        )

    # Assign UFS speaker labels to AA transcript words & sentences
    transcript_aa = give_speaker_labels(
        transcript_aa, 
        processed_ufs, 
        score_assignments,
    )
    transcript_aa = build_final_sentences(
        transcript_aa,
    )

    # Save output
    save_aa(s, e, transcript_aa)


# Tests & dev

In [85]:
s = "7"

import os
e_list = sorted(glob.glob(f"/home/mstlaure/Documents/Marie/neuromod/friends_annotations/annotation_results/Speech2Text/s{s}/*wUtter_transcript.json"))

t_dict = {}
fails = []
for epath in e_list:
    e = os.path.basename(epath).split("_")[1][-3:]
    
    try:
        processed_utterances = prep_ufs(s, e[:2])
        aa_transcript = prep_aa(s, e)
        
        ancs, tops = find_anchors(
            aa_transcript,
            processed_utterances,
            verbose=False,
        )
        
        anchor_idx, score_assignments = process_chunks(
            ancs,
            tops,
            aa_transcript,
            processed_utterances,   
            verbose=False,
        )
        
        for c in range(6):
            #print(f"ITERATION {c+1}")
            aa_transcript, score_assignments = finetune_segments( 
                aa_transcript,
                processed_utterances,   
                score_assignments,
                verbose=False,
            )
        aa_transcript = give_speaker_labels(aa_transcript, processed_utterances, score_assignments)
        t_dict[e] = {
            "transcript": aa_transcript,
            "scores": score_assignments,
            "utter": processed_utterances,
        }                
        # Visual QC after assigning speaker labels
        """
        print(len(aa_transcript['sentences']), score_assignments.shape)
        
        for i in range(0, score_assignments.shape[0]):
            i_idx = score_assignments[i][0]
            sen = aa_transcript['sentences'][i_idx]
        
            j_idx = score_assignments[i][1]
            utt = processed_utterances[j_idx]
            
            print(i, i_idx, f"{sen['speaker']}: {sen['text']}")
            print(j_idx, f"{utt[0]}: {utt[1]}")
            print()
        """
    except:
        print(f"Could not process s0{s}e{e}")
        fails.append(f"s0{s}e{e}")

In [86]:
len(t_dict.keys()), len(e_list), t_dict.keys(), fails

(49,
 49,
 dict_keys(['01a', '01b', '02a', '02b', '03a', '03b', '04a', '04b', '05a', '05b', '06a', '06b', '07a', '07b', '08a', '08b', '09a', '09b', '10a', '10b', '11a', '11b', '12a', '12b', '13a', '13b', '14a', '14b', '15a', '15b', '16a', '16b', '16c', '17a', '17b', '18a', '18b', '19a', '19b', '20a', '20b', '21a', '21b', '22a', '22b', '23a', '23b', '23c', '23d']),
 [])

In [49]:
epi = "01a"
aa_t = t_dict[epi]["transcript"]
p_utter = t_dict[epi]["utter"]
scores = t_dict[epi]["scores"]

print(len(aa_t['sentences']), scores.shape)

for i in range(0, scores.shape[0]):
    i_idx = scores[i][0]
    sen = aa_t['sentences'][i_idx]

    j_idx = scores[i][1]
    utt = p_utter[j_idx]
    
    print(i, i_idx, f"{sen['speaker']}: {sen['text']}")
    print(j_idx, f"{utt[0]}: {utt[1]}")
    print()    

215 (215, 4)
0 0 joey: I'm telling you, that girl totally winked at me.
0 joey:  i'm tellin' ya that girl totally winked at me.

1 1 all: Did not wink at you.
1 all:  did not, she did not wink at you...

2 2 chandler: Huh?
3 chandler:  huh.

3 3 ross: I have to say Tupelo Honey by Van Morrison.
4 ross:  i have to say tupolo honey by van morrison.

4 4 rachel: No way.
5 rachel:  nooo way!

5 5 rachel: The most romantic song ever was the Way We Were.
6 rachel: the most romantic song ever is the way we were.

6 6 phoebe: See, I think the one that Elton John wrote for that guy on who's the Boss?
7 phoebe:  see, i-i think that one that elton john wrote for, um, that guy on who's the boss.

7 7 rachel: What song is that, Pheebs?
8 rachel:  what song was that, pheebs?

8 8 phoebe: Hold me close Young Tony Danza
9 phoebe:  hold me close, young tony dan-za.

9 9 phoebe: hi, Monica. Hi.
10 phoebe:  hi monica!

10 10 ross: Hey, Ma.
11 ross:  hey mon!

11 11 rachel: Hey, Ma.
12 rachel:  hey mon!



In [141]:
s = "7"
e = "23d"

processed_utterances = prep_ufs(s, e[:2])
aa_transcript = prep_aa(s, e)

ancs, tops = find_anchors(
    aa_transcript,
    processed_utterances,
    verbose=False,
)

anchor_idx, score_assignments = process_chunks(
    ancs,
    tops,
    aa_transcript,
    processed_utterances,   
    verbose=False,
)

for c in range(6):
    print(f"ITERATION {c+1}")
    aa_transcript, score_assignments = finetune_segments( 
        aa_transcript,
        processed_utterances,   
        score_assignments,
    )

aa_transcript = give_speaker_labels(aa_transcript, processed_utterances, score_assignments)

# Visual QC after assigning speaker labels
print(len(aa_transcript['sentences']), score_assignments.shape)

for i in range(0, score_assignments.shape[0]):
    i_idx = score_assignments[i][0]
    sen = aa_transcript['sentences'][i_idx]

    j_idx = score_assignments[i][1]
    utt = processed_utterances[j_idx]
    
    print(i, i_idx, f"{sen['speaker']}: {sen['text']}")
    print(j_idx, f"{utt[0]}: {utt[1]}")
    print()


ITERATION 1
ITERATION 2
ITERATION 3
ITERATION 4
ITERATION 5
ITERATION 6
199 (199, 4)
0 0 chandler: Oh, fresh air.
839 chandler:  oh fresh air!

1 1 rachel: You're going to cover me?
840 rachel:  you're going to cover for me.

2 2 phoebe: Wait,
841 phoebe:  wait!

3 3 phoebe: maybe, maybe you're overreacting.
842 phoebe: maybe, maybe you’re overreacting!

4 4 phoebe: You do that, you know,
843 phoebe: you do that y’know.

5 5 rachel: Phoebe, we gotta do something.
844 rachel:  well phoebe, we gotta do something!

6 6 rachel: You know,
845 rachel: well, y’know.

7 7 rachel: I mean there's no way Joey's gonna make it in time.
846 rachel: i mean there’s no way joey’s gonna make it in time.

8 8 rachel: So I'm go through the hotel and see if there's any other weddings going on.
847 rachel: so i’m gonna through the hotel and see if there’s any other weddings going on.

9 9 phoebe: Okay.
848 phoebe:  okay.

10 10 phoebe: Oh, but don't tell them Monica's pregnant cause they frown on that.
849 

In [142]:
print(s, e)
print(len(aa_transcript['sentences']))

aa_transcript = build_final_sentences(
    aa_transcript,
)

# Visual QC after rebuilding sentences w matching speakers
print(len(aa_transcript['sentences']))

for i in range(len(aa_transcript['sentences'])):
    print(i, f"{aa_transcript['sentences'][i]['speaker']}: {aa_transcript['sentences'][i]['text']}")
    print()


7 23d
199
139
0 chandler: Oh, fresh air.

1 rachel: You're going to cover me?

2 phoebe: Wait, maybe, maybe you're overreacting. You do that, you know,

3 rachel: Phoebe, we gotta do something. You know, I mean there's no way Joey's gonna make it in time. So I'm go through the hotel and see if there's any other weddings going on.

4 phoebe: Okay. Oh, but don't tell them Monica's pregnant cause they frown on that. Okay. Bye. Bye.

5 rachel: Anastasakis Papa Sifakis wedding. Excellent.

6 rachel: Congratulations. Mazel tov. Hi. Oh, great hat.

7 rachel: Listen, I need you to perform another wedding. Can you do that?

8 the priest: I don't know. Are they Greek orthodox?

9 rachel: Yeah, yeah, they're my friends. Monica Stephanopoulos and Chandler Acidopoulos.

10 ross: Hi. Hi.

11 ross: Hi.

12 ross: Has anyone seen Chandler?

13 mr. geller: I thought he was with you.

14 ross: He was with me. Um, we're playing a little game. You know, hide and seek.

15 mr. geller: You can't ask us, son.

In [143]:
print(f"s0{s}e{e}")
print(len(aa_transcript['sentences']))


s07e23d
139


In [144]:

save_aa(s, e, aa_transcript)

In [84]:
# Check before finetuning
print(len(aa_transcript['sentences']), score_assignments.shape)
for i in range(0, score_assignments.shape[0]):
    print(i, score_assignments[i][0], aa_transcript['sentences'][score_assignments[i][0]]['text'])
    print(score_assignments[i][1], processed_utterances[score_assignments[i][1]][1])
    print()


227 (227, 4)
0 0 Hey, what's going on?
0  hey, what’s going on?

1 1 I found a note on my door.
2  i found a note on my door, "come to monica’s quick, bring champagne and a three musketeers bar."

2 2 Come to Monica's.
2  i found a note on my door, "come to monica’s quick, bring champagne and a three musketeers bar."

3 3 Quick, bring champagne and a three Musketeers bar.
2  i found a note on my door, "come to monica’s quick, bring champagne and a three musketeers bar."

4 4 Yeah, I'll take that.
3  yeah i’ll take that.

5 5 What's up?
4  what’s up?

6 6 Monica and I are engaged.
5  monica and i are engaged.

7 7 Oh my God.
6  oh my god.

8 8 Congratulations.
7 congratulations.

9 9 Where is she?
9  where is she?

10 10 I'm engaged.
10  i’m engaged!!!!!!

11 11 I'm engaged.
10  i’m engaged!!!!!!

12 12 Yeah, she's been out there for 20 minutes.
12  yeah, she’s been out there for twenty minutes, i’m surprised you didn’t hear her on the way over.

13 13 I'm surprised you didn't hear her 

In [77]:
s, e

('3', '01b')

In [78]:
for k in range(len(score_assignments)):
    if k > 0:
        i1, j1, s1, l1 = score_assignments[k-1]
        i2, j2, s2, l2 = score_assignments[k]
        if j1 > j2:
            print(i2, j2-j1)
        if j2 - j1 > 2:
            print(i2, j2-j1)
            

44 7
113 5
124 3
131 6
132 3
136 3
203 3


In [79]:
len(score_assignments)

235

In [80]:

for c in range(6):
    print(f"ITERATION {c+1}")
    aa_transcript, score_assignments = finetune_segments( 
        aa_transcript,
        processed_utterances,   
        score_assignments,
    )


ITERATION 1
ITERATION 2
ITERATION 3
ITERATION 4
ITERATION 5
ITERATION 6


In [81]:
# Check after finetuning
print(len(aa_transcript['sentences']), score_assignments.shape)
for i in range(0, score_assignments.shape[0]):
    print(i, score_assignments[i][0], aa_transcript['sentences'][score_assignments[i][0]]['text'])
    print(score_assignments[i][1], processed_utterances[score_assignments[i][1]][1])
    print()


181 (181, 4)
0 0 Be like a private, personal thing between us.
222  look, that was supposed to be like a private, personal thing between us.

1 1 Okay? Ross, Phoebe is my girlfriend, okay? We tell each other everything,
223  okay, ross, phoebe is my girlfriend, okay, we tell each other everything.

2 2 you know? I mean, come on. Guys do the same thing. But what about that locker room stuff?
224 you know, i mean, come on, guys do the same thing, i mean, what about all that locker room stuff.

3 3 That's different, okay?
225  that's different, okay.

4 4 That's like, who dated a stripper? Or who did it on the back of the Staten island ferry?
226 that's like, uh 'who dated a stripper?' or 'who did it on the back of the staton island ferry?'.

5 5 Both of those. Joey.
227  were both of those joey?

6 6 Yeah,
228  yeah.

7 7 but, you know, you don't talk about, like, you know, your girlfriend and the intimate stuff you do with her.
229 look, you don't, you don't talk about like, you know, y

In [82]:
for k in range(len(score_assignments)):
    if k > 0:
        i1, j1, s1, l1 = score_assignments[k-1]
        i2, j2, s2, l2 = score_assignments[k]
        if j2 - j1 > 1:
            print(i2, j2-j1)

14 2
89 2
94 2
95 2
101 2
109 2
114 2
171 2
174 2


In [58]:
aa_transcript = give_speaker_labels(aa_transcript, processed_utterances, score_assignments)


In [59]:

# Visual QC after assigning speaker labels
print(len(aa_transcript['sentences']), score_assignments.shape)

for i in range(0, score_assignments.shape[0]):
    i_idx = score_assignments[i][0]
    sen = aa_transcript['sentences'][i_idx]

    j_idx = score_assignments[i][1]
    utt = processed_utterances[j_idx]
    
    print(i, i_idx, f"{sen['speaker']}: {sen['text']}")
    print(j_idx, f"{utt[0]}: {utt[1]}")
    print()



216 (216, 4)
0 0 joey: I'm telling you, that girl totally winked at me.
0 joey:  i'm tellin' ya that girl totally winked at me.

1 1 all: Did not wink at you.
1 all:  did not, she did not wink at you...

2 2 chandler: Huh?
3 chandler:  huh.

3 3 ross: I have to say Tupelo Honey by Van Morrison.
4 ross:  i have to say tupolo honey by van morrison.

4 4 rachel: No way.
5 rachel:  nooo way!

5 5 rachel: The most romantic song ever was the Way We Were.
6 rachel: the most romantic song ever is the way we were.

6 6 phoebe: See, I think the one that Elton John wrote for that guy on who's the Boss?
7 phoebe:  see, i-i think that one that elton john wrote for, um, that guy on who's the boss.

7 7 rachel: What song is that, Pheebs?
8 rachel:  what song was that, pheebs?

8 8 phoebe: Hold me close Young Tony Danza
9 phoebe:  hold me close, young tony dan-za.

9 9 phoebe: hi, Monica. Hi.
10 phoebe:  hi monica!

10 10 ross: Hey, Ma.
11 ross:  hey mon!

11 11 rachel: Hey, Ma.
12 rachel:  hey mon!



In [60]:
print(s, e)

3 01a


In [61]:
print(len(aa_transcript['sentences']))

216


In [62]:
aa_transcript = build_final_sentences(
    aa_transcript,
)

In [63]:

# Visual QC after rebuilding sentences w matching speakers
print(len(aa_transcript['sentences']))

for i in range(len(aa_transcript['sentences'])):
    print(i, f"{aa_transcript['sentences'][i]['speaker']}: {aa_transcript['sentences'][i]['text']}")
    print()

#print(len(aa_transcript['sentences']))


156
0 joey: I'm telling you, that girl totally winked at me.

1 all: Did not wink at you.

2 chandler: Huh?

3 ross: I have to say Tupelo Honey by Van Morrison.

4 rachel: No way. The most romantic song ever was the Way We Were.

5 phoebe: See, I think the one that Elton John wrote for that guy on who's the Boss?

6 rachel: What song is that, Pheebs?

7 phoebe: Hold me close Young Tony Danza

8 phoebe: hi, Monica. Hi.

9 ross: Hey, Ma.

10 rachel: Hey, Ma.

11 phoebe: Oh, my God. Has she slept at all?

12 ross: No.

13 rachel: No, it's been three nights in a row.

14 ross: Yeah, she finally stopped crying yesterday. But then she found one of Richard's cigar butts out on the terrace.

15 phoebe: Okay, that explains it. I got a call at 2 in the morning and all I could hear was, like, this high squeaky sound. So I thought, oh, okay, so it's like a mouse or a possum.

16 phoebe: Then I realized, like, okay, where would a mouse or a possum get the money to make the phone call?

17 chandler:

# Scripts graveyard

"Oh well, that sure didn't work!"

In [ ]:

def split_rec(
    temp_wlist, 
    p_utter, 
    split_indices, 
    j1,
    j2,
    w_start, 
    w_stop, 
    k_0, 
    k_n,
):
    """."""
    j_0 = max(0, j1-1)
    j_n = min(j2+2, len(p_utter))

    for k in range(k_0, k_n):
        targ1 = " ".join([u[1] for u in p_utter[j_0:j1+k]])
        targ2 = " ".join([u[1] for u in p_utter[j1+k:j_n]])

        split_candidates = []
        split_idx, score = find_best_split(
            temp_wlist, 
            targ1, 
            targ2,
            i_start=w_start, 
            i_stop=w_stop + 1,
        )    
        split_candidates.append((k, split_idx, score))
    
    best_k, best_split, best_score = split_candidates[np.argmax(
        [x[2] for x in split_candidates])]

    if best_k == k_0 or best_k == (k_n - 1):
        split_indices[best_k] = split_idx
    if best_k > k_0:
        split_indices = split_rec(
            temp_wlist, 
            p_utter, 
            split_indices, 
            j1, 
            j2,
            w_start,
            split_idx, 
            k_0, 
            best_k,        
        )
    if best_k < (k_n - 1):
        split_indices = split_rec(
            temp_wlist, 
            p_utter, 
            split_indices, 
            j1, 
            j2,
            split_idx, 
            w_stop, 
            best_k+1, 
            k_n,        
        )

    return split_indices


def gap_segs_dev(aa_t, p_utter, i, j1, j2):
    # compact four consecutive sentences for more context
    i_0 = max(0, i - 1)
    i_n = min(i+3, len(aa_t['sentences']))
    temp_wlist = [w for s in aa_t['sentences'][i_0:i_n] for w in s["words"]]

    # Set search space boundaries: limit splits to two middle sentences i and i+1
    w_start = 0 if i == 0 else len(aa_t['sentences'][i_0]['words'])
    w_stop = len([w for s in aa_t['sentences'][i_0:i+2] for w in s["words"]])
    
    """
    Make it recursive
    """
    split_indices = {}
    k_0 = 1
    k_n = j2-j1+1

    split_indices = split_rec(
        temp_wlist, 
        p_utter, 
        split_indices, 
        j1, 
        j2,
        w_start, 
        w_stop, 
        k_0, 
        k_n,        
    )
        
    # merge two middle sentences
    aa_t = merge_segs(aa_t, i)

    cutwords = 0
    n_cuts = 0
    prev_cuts = {w_start, w_stop}
    j_vals = []    
    for k in range(k_0, k_n):
        s = split_indices[k]
        if s not in prev_cuts:
            w2cut = s-(w_start+cutwords)
            aa_t = split_seg(aa_t, i+n_cuts, w2cut)
            j_vals.append(j1+k-1)
            prev_cuts.add(s)
            cutwords += w2cut
            n_cuts += 1
    j_vals.append(j2)    
                    
    return aa_t, j_vals



In [ ]:
def process_chunks_dev(
    ancs,
    tops,
    aa_transcript,
    processed_utterances,
    verbose=False,
):
    """."""
    score_assignments = np.full(
        (len(tops), 4), -1, dtype=int,
    )
    anchor_idx = np.array(tops)[ancs]    
    for i in range(anchor_idx.shape[0]): 
        
        if i == anchor_idx.shape[0] - 1:
            # skip assigning sentences from last segment because unbound
            pass    
        else:
            assign_chunk(
                anchor_idx[i][0],          # include current anchor speech-to-text in search
                anchor_idx[i+1][0],        # exclude next anchor speech-to-text from search
                anchor_idx[i][1],         # include current anchor's ccf match in search 
                anchor_idx[i+1][1] + 1,   # include next anchor's ccf match in search, in case precedent text also matches that utterance  
                aa_transcript,
                processed_utterances, 
                score_assignments,
                verbose=verbose,
            )
    
    # assign sentences from first segment
    assign_top(
        anchor_idx[0],
        aa_transcript,
        processed_utterances, 
        score_assignments,
    )
    # assign sentences from last segment
    assign_btm(
        anchor_idx[-1],
        aa_transcript,
        processed_utterances, 
        score_assignments,
    )
    
    return anchor_idx, score_assignments


def assign_top(
    start_vals,
    aa_transcript,
    processed_utterances, 
    score_assignments,
):
    """
    Starting from first anchor, work backward to assign speech2text sentences to transcript utterances 
    """
    if start_vals[0] == 0:
        """
        Edge case for which the first anchor is also the first speech2text sentence
        No unassigned sentences remain at the top, nothing to do
        """
        return score_assignments
        
    else:
        j = int(start_vals[1])        
        
        for i in range(start_vals[0]-1, -1, -1):
        
            # test 1: concat sentence with sentence below and calculate fit w sentence below's match
            t1_score = (j, string_match(
                aa_transcript['sentences'][i]['text'] + " " + aa_transcript['sentences'][i+1]['text'],
                processed_utterances[j][1],
            ))
        
            # test 2: calculate sentence fit with utterance just before next sentence's match
            t2_score = (max(0, j-1), string_match(
                aa_transcript['sentences'][i]['text'],
                processed_utterances[max(0, j-1)][1],
            ))
        
            # test 3: concat sentence with sentence above, and calculate fit with utterance just before next sentence's match
            t3_score = (0, (0, 0)) if i == 0 else (max(0, j-1), string_match(
                aa_transcript['sentences'][i-1]['text'] + " " + aa_transcript['sentences'][i]['text'],
                processed_utterances[max(0, j-1)][1],
            ))
        
            # test 4: calculate sentence fit with utterance two before next sentence's match
            t4_score = (max(0, j-2), string_match(
                aa_transcript['sentences'][i]['text'],
                processed_utterances[max(0, j-2)][1],
            ))
        
            # test 5: concat sentence with sentence above, and calculate fit with utterance two before next sentence's match 
            t5_score = (0, (0, 0)) if i == 0 else (max(0, j-2), string_match(
                aa_transcript['sentences'][i-1]['text'] + " " + aa_transcript['sentences'][i]['text'],
                processed_utterances[max(0, j-2)][1],
            ))
        
            res_list = [t1_score, t2_score, t3_score, t4_score, t5_score]
            
            # re-assigns j to best score j, becomes "previous" as moves backward
            j, (s, l) = res_list[int(np.argmax([x[1][0] for x in res_list]))]
        
            score_assignments[i] = np.array([i, j, s, l])        
    
        return score_assignments


def assign_btm(
    start_vals,
    aa_transcript,
    processed_utterances, 
    score_assignments,
):
    """
    Starting from last anchor, work forward to assign speech2text sentences to transcript utterances 
    """
    # Assign last anchor to scores
    score_assignments[start_vals[0]] = start_vals
    
    i_lim = score_assignments.shape[0] - 1
    if start_vals[0] == i_lim:
        """
        Edge case for which the last anchor is also the last speech2text sentence
        No unassigned sentences remain at the bottom, nothing to do
        """
        return score_assignments
        
    else:
        j = int(start_vals[1])
        j_lim = len(processed_utterances) - 1

        for i in range(start_vals[0]+1, i_lim+1, 1):

            # test 1: concat sentence with sentence above and calculate fit w sentence above's match
            t1_score = (j, string_match(
                aa_transcript['sentences'][i-1]['text'] + " " + aa_transcript['sentences'][i]['text'],
                processed_utterances[j][1],
            ))

            # test 2: calculate sentence fit with utterance just after previous sentence's match
            t2_score = (min(j_lim, j+1), string_match(
                aa_transcript['sentences'][i]['text'],
                processed_utterances[min(j_lim, j+1)][1],
            ))

            # test 3: concat sentence with sentence below, and calculate fit with utterance just after previous sentence's match
            t3_score = (0, (0, 0)) if i == i_lim else (min(j_lim, j+1), string_match(
                aa_transcript['sentences'][i]['text'] + " " + aa_transcript['sentences'][i+1]['text'],
                processed_utterances[min(j_lim, j+1)][1],
            ))
        
            # test 4: calculate sentence fit with utterance two after previous sentence's match
            t4_score = (min(j_lim, j+2), string_match(
                aa_transcript['sentences'][i]['text'],
                processed_utterances[min(j_lim, j+2)][1],
            ))
        
            # test 5: concat sentence with sentence below, and calculate fit with utterance two after previous sentence's match         
            t5_score = (0, (0, 0)) if i == i_lim else (min(j_lim, j+2), string_match(
                aa_transcript['sentences'][i]['text'] + " " + aa_transcript['sentences'][i+1]['text'],
                processed_utterances[min(j_lim, j+2)][1],
            ))
        
            res_list = [t1_score, t2_score, t3_score, t4_score, t5_score]
            
            # re-assigns j to best score j, becomes "previous" as moves forward
            j, (s, l) = res_list[int(np.argmax([x[1][0] for x in res_list]))]
        
            score_assignments[i] = np.array([i, j, s, l])        
    
        return score_assignments

In [ ]:

def print_gapinfo(
    i1, 
    i2, 
    j1, 
    j2, 
    aa_t, 
    p_utter,    
):
    print(aa_t['sentences'][i1]['text'])
    print(aa_t['sentences'][i2]['text'])
    print()
    print(p_utter[j1][1])
    print(p_utter[j2][1])
    print()
    for m in range (j1, j2+1):
        print(p_utter[m][1])
        

def finetune_segments_dev(
    anchor_idx,
    aa_transcript,
    processed_utterances,
    score_assignments,
):
    """."""
    i = 1
    #i = anchor_idx[0][0]
    while i < len(aa_transcript['sentences']):
        if i + 1 == len(aa_transcript['sentences']):
            # Last segment, just exit loop
            i += 1

        else:
            j1 = score_assignments[i][1]
            j2 = score_assignments[i+1][1]
            assert j1 <= j2
    
            # two subsequent sentences w same match: merge sentences
            if j1 == j2:
                aa_transcript = merge_segs(
                    aa_transcript, i,
                )
                s, l = string_match(
                    aa_transcript['sentences'][i]['text'],
                    processed_utterances[j1][1],
                )
                score_assignments[i] = np.array([i, j1, s, l])
                if i + 2 == len(aa_transcript['sentences']):
                    score_assignments = score_assignments[:-1]
                else:
                    score_assignments[i+2:, 0] = score_assignments[i+2:, 0] - 1
                    score_assignments = np.concatenate(
                        (score_assignments[:i+1], score_assignments[i+2:])
                    )
                # i+= 0  # Do NOT update i, sentence list shortened by one 
    
            # Cap gap size at 10... otherwise most likely wrong match... 
            elif j2 > j1 and j2 - j1 < 7:
                aa_transcript, j_vals = gap_segs(
                    aa_transcript, processed_utterances, i, j1, j2,
                )
                temp_scores = []
                for k in range(len(j_vals)):
                    s, l = string_match(
                        aa_transcript['sentences'][i+k]['text'],
                        processed_utterances[j_vals[k]][1],
                    )
                    temp_scores.append([i+k, j_vals[k], s, l])
                if i + 2 == len(aa_transcript['sentences']):
                    score_assignments = np.concatenate(
                        (score_assignments[:i], np.array(temp_scores))
                    )
                else:
                    score_assignments[i+2:, 0] = score_assignments[i+2:, 0] + len(j_vals) - 2
                    score_assignments = np.concatenate(
                        (score_assignments[:i], np.array(temp_scores), score_assignments[i+2:])
                    )
                i += len(j_vals) - 1    
            
            else:
                # Handle large gaps... merge two next consecutive segments, assign second seg's match to scores, and re-try
                print(f"Warning! Large jump of {j2 - j1} in matching indices between lines {i} and {i+1}")
                """
                print_gapinfo(
                    i, i+1, j1, j2, 
                    aa_transcript, 
                    processed_utterances,
                )
                """
            
                if i + 2 < len(aa_transcript['sentences']):
                    new_j2 = score_assignments[i+2][1]
                    aa_transcript = merge_segs(
                        aa_transcript, i+1,
                    )
                    s, l = string_match(
                        aa_transcript['sentences'][i+1]['text'],
                        processed_utterances[new_j2][1],
                    )
                    score_assignments[i+1] = np.array([i+1, new_j2, s, l])
                    if i + 3 == len(aa_transcript['sentences']):
                        score_assignments = score_assignments[:-1]
                    else:
                        score_assignments[i+3:, 0] = score_assignments[i+3:, 0] - 1
                        score_assignments = np.concatenate(
                            (score_assignments[:i+2], score_assignments[i+3:])
                        )
                        
                else:
                    i += 1
    
    return aa_transcript, score_assignments

